In [74]:
import pandas as pd
import numpy as np
import seaborn as sns
from sklearn.impute import KNNImputer

In [75]:
train = pd.read_csv("/content/drive/MyDrive/Kaggle/Sapceship Titanic/train.csv")
test  = pd.read_csv("/content/drive/MyDrive/Kaggle/Sapceship Titanic/test.csv")

test['Transported'] = False

df = pd.concat([train,test], sort = False)

df.drop(["Name", "PassengerId"], axis=1,inplace=True)

df.head(5)

,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Transported
0,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,False
1,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,True
2,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,False
3,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,False
4,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,True


In [76]:
df.shape[0] == train.shape[0] + test.shape[0]

True

In [77]:
df.isna().sum()

,0
HomePlanet,288
CryoSleep,310
Cabin,299
Destination,274
Age,270
VIP,296
RoomService,263
FoodCourt,289
ShoppingMall,306
Spa,284


In [78]:
df[['Deck','Num','Side']] = df['Cabin'].str.split('/', expand=True)

df = df.drop(columns = ['Cabin'])

df['Deck'] = df['Deck'].fillna("U")
df["Num"] = df["Num"].fillna(-1)
df['Side'] = df['Side'].fillna('U')

df['Destination'].value_counts()

,count
Destination,
TRAPPIST-1e,8871
55 Cancri e,2641
PSO J318.5-22,1184


In [79]:
df['Deck'] = df['Deck'].map({'G' : 0, 'F' : 1, 'E' : 2,
                             'D' : 3, 'C' : 4, 'B' : 5,
                             'A' : 6, 'U' : 7, 'T' : 8})
df['Side'] = df['Side'].map({'U' : -1, 'P' : 1, 'S' : 2})

In [80]:
df.head(5)

,HomePlanet,CryoSleep,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Transported,Deck,Num,Side
0,Europa,False,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,False,5,0,1
1,Earth,False,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,True,1,0,2
2,Europa,False,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,False,6,0,2
3,Europa,False,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,False,6,0,2
4,Earth,False,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,True,1,1,2


In [81]:
df['HomePlanet'].value_counts()

,count
HomePlanet,
Earth,6865
Europa,3133
Mars,2684


In [82]:
impute_list = ['Age', 'VIP', 'Num', 'CryoSleep', 'Side',
               'Deck', 'RoomService', 'FoodCourt',
               'ShoppingMall', 'Spa', 'VRDeck']

rest = list( set(df.columns) - set(impute_list) )

restdf = df[rest]
imp = KNNImputer()

imputeddf = imp.fit_transform(df[impute_list])
imputeddf = pd.DataFrame(imputeddf, columns = impute_list)

df = pd.concat([restdf.reset_index(drop=True),
                imputeddf.reset_index(drop=True)],
               axis=1)

df.head()

,Transported,Destination,HomePlanet,Age,VIP,Num,CryoSleep,Side,Deck,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck
0,False,TRAPPIST-1e,Europa,39.0,0.0,0.0,0.0,1.0,5.0,0.0,0.0,0.0,0.0,0.0
1,True,TRAPPIST-1e,Earth,24.0,0.0,0.0,0.0,2.0,1.0,109.0,9.0,25.0,549.0,44.0
2,False,TRAPPIST-1e,Europa,58.0,1.0,0.0,0.0,2.0,6.0,43.0,3576.0,0.0,6715.0,49.0
3,False,TRAPPIST-1e,Europa,33.0,0.0,0.0,0.0,2.0,6.0,0.0,1283.0,371.0,3329.0,193.0
4,True,TRAPPIST-1e,Earth,16.0,0.0,1.0,0.0,2.0,1.0,303.0,70.0,151.0,565.0,2.0


In [83]:
df['HomePlanet'] = df['HomePlanet'].fillna('U')
df['Destination'] = df['Destination'].fillna('U')
category_colls = ['HomePlanet', 'Destination']

for col in category_colls:
  df = pd.concat([df, pd.get_dummies(df[col], prefix = col)],
                 axis=1)

df.head()

,Transported,Destination,HomePlanet,Age,VIP,Num,CryoSleep,Side,Deck,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,HomePlanet_Earth,HomePlanet_Europa,HomePlanet_Mars,HomePlanet_U,Destination_55 Cancri e,Destination_PSO J318.5-22,Destination_TRAPPIST-1e,Destination_U
0,False,TRAPPIST-1e,Europa,39.0,0.0,0.0,0.0,1.0,5.0,0.0,0.0,0.0,0.0,0.0,False,True,False,False,False,False,True,False
1,True,TRAPPIST-1e,Earth,24.0,0.0,0.0,0.0,2.0,1.0,109.0,9.0,25.0,549.0,44.0,True,False,False,False,False,False,True,False
2,False,TRAPPIST-1e,Europa,58.0,1.0,0.0,0.0,2.0,6.0,43.0,3576.0,0.0,6715.0,49.0,False,True,False,False,False,False,True,False
3,False,TRAPPIST-1e,Europa,33.0,0.0,0.0,0.0,2.0,6.0,0.0,1283.0,371.0,3329.0,193.0,False,True,False,False,False,False,True,False
4,True,TRAPPIST-1e,Earth,16.0,0.0,1.0,0.0,2.0,1.0,303.0,70.0,151.0,565.0,2.0,True,False,False,False,False,False,True,False


In [84]:
df = df.drop(columns = category_colls)

In [85]:
df.head()

,Transported,Age,VIP,Num,CryoSleep,Side,Deck,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,HomePlanet_Earth,HomePlanet_Europa,HomePlanet_Mars,HomePlanet_U,Destination_55 Cancri e,Destination_PSO J318.5-22,Destination_TRAPPIST-1e,Destination_U
0,False,39.0,0.0,0.0,0.0,1.0,5.0,0.0,0.0,0.0,0.0,0.0,False,True,False,False,False,False,True,False
1,True,24.0,0.0,0.0,0.0,2.0,1.0,109.0,9.0,25.0,549.0,44.0,True,False,False,False,False,False,True,False
2,False,58.0,1.0,0.0,0.0,2.0,6.0,43.0,3576.0,0.0,6715.0,49.0,False,True,False,False,False,False,True,False
3,False,33.0,0.0,0.0,0.0,2.0,6.0,0.0,1283.0,371.0,3329.0,193.0,False,True,False,False,False,False,True,False
4,True,16.0,0.0,1.0,0.0,2.0,1.0,303.0,70.0,151.0,565.0,2.0,True,False,False,False,False,False,True,False


In [86]:
#feature engineering
bill_cols = ['RoomService', 'FoodCourt', 'ShoppingMall',
             'Spa', 'VRDeck']

df['amt_spent'] = df[bill_cols].sum(axis = 1)
df['std_amt_spent'] = df[bill_cols].std(axis = 1)
df['mean_amt_spent'] = df[bill_cols].mean(axis = 1)

df['3_high_cols'] = df['CryoSleep'] + df['HomePlanet_Europa'] + df['Destination_55 Cancri e']
df['3_low_cols'] = df['mean_amt_spent'] + df['amt_spent'] + df['HomePlanet_Earth']

df.head()

,Transported,Age,VIP,Num,CryoSleep,Side,Deck,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,HomePlanet_Earth,HomePlanet_Europa,HomePlanet_Mars,HomePlanet_U,Destination_55 Cancri e,Destination_PSO J318.5-22,Destination_TRAPPIST-1e,Destination_U,amt_spent,std_amt_spent,mean_amt_spent,3_high_cols,3_low_cols
0,False,39.0,0.0,0.0,0.0,1.0,5.0,0.0,0.0,0.0,0.0,0.0,False,True,False,False,False,False,True,False,0.0,0.000000,0.0,1.0,0.0
1,True,24.0,0.0,0.0,0.0,2.0,1.0,109.0,9.0,25.0,549.0,44.0,True,False,False,False,False,False,True,False,736.0,227.807375,147.2,0.0,884.2
2,False,58.0,1.0,0.0,0.0,2.0,6.0,43.0,3576.0,0.0,6715.0,49.0,False,True,False,False,False,False,True,False,10383.0,3013.383198,2076.6,1.0,12459.6
3,False,33.0,0.0,0.0,0.0,2.0,6.0,0.0,1283.0,371.0,3329.0,193.0,False,True,False,False,False,False,True,False,5176.0,1373.410427,1035.2,1.0,6211.2
4,True,16.0,0.0,1.0,0.0,2.0,1.0,303.0,70.0,151.0,565.0,2.0,True,False,False,False,False,False,True,False,1091.0,223.988169,218.2,0.0,1310.2


In [87]:
df.corr()['Transported'].sort_values(ascending = False)

,Transported
Transported,1.000000
CryoSleep,0.324335
3_high_cols,0.284152
HomePlanet_Europa,0.131977
Destination_55 Cancri e,0.083625
Deck,0.077959
Side,0.059872
FoodCourt,0.034746
HomePlanet_U,0.006403
HomePlanet_Mars,0.005643


In [88]:
train, test = df[ : train.shape[0]], df[train.shape[0] : ]
test = test.drop(columns = 'Transported')
train.shape, test.shape

((8693, 25), (4277, 24))

In [89]:
from xgboost import XGBClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from lightgbm import LGBMClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

In [90]:
x = train.drop(columns='Transported')
y = train['Transported']

xtrain, xtest,ytrain,ytest = train_test_split(x,y,
                                              test_size=0.2,
                                              random_state=42)

model_1 = LogisticRegression()
model_2 = DecisionTreeClassifier()
model_3 = RandomForestClassifier()
model_4 = XGBClassifier()
model_5 = LGBMClassifier()



In [91]:
model_1.fit(xtrain,ytrain)
pred = model_1.predict(xtest)
accuracy_score(ytest,pred)


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


0.7751581368602645

In [92]:
model_2.fit(xtrain,ytrain)
pred = model_2.predict(xtest)
accuracy_score(ytest,pred)

0.7556066705002875

In [93]:
model_3.fit(xtrain,ytrain)
pred = model_3.predict(xtest)
accuracy_score(ytest,pred)

0.7912593444508338

In [94]:
model_4.fit(xtrain,ytrain)
pred = model_4.predict(xtest)
accuracy_score(ytest,pred)

0.7832087406555491

In [95]:
model_5.fit(xtrain,ytrain)
pred = model_5.predict(xtest)
accuracy_score(ytest,pred)

[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Number of positive: 3500, number of negative: 3454
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001480 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2704
[LightGBM] [Info] Number of data points in the train set: 6954, number of used features: 24
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.503307 -> initscore=0.013230
[LightGBM] [Info] Start training from score 0.013230


0.7918343875790684

In [96]:
df_dummy = pd.read_csv("/content/drive/MyDrive/Kaggle/Sapceship Titanic/test.csv")
pred = model_5.predict(test)

final = pd.DataFrame()
final['PassengerId'] = df_dummy['PassengerId']
final['Transported'] = pred

In [97]:
final.to_csv('submission.csv', index = False)